In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
%cd /kaggle/working
!git clone https://github.com/Djuybu/r2AI_2026


/kaggle/working
Cloning into 'r2AI_2026'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 19 (delta 0), reused 19 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 8.98 KiB | 4.49 MiB/s, done.


In [2]:
%cd /kaggle/working/r2AI_2026

/kaggle/working/r2AI_2026


In [3]:
# Cài đặt các gói phụ thuộc trực tiếp với phiên bản khống chế tương thích Kaggle
!pip install \
    "protobuf>=5.26.1,<6.0dev" \
    "starlette>=0.40.0,<1.0.0" \
    "opentelemetry-api>=1.35.0,<1.39.0" \
    "opentelemetry-sdk>=1.35.0,<1.39.0" \
    "numba>=0.60.0,<0.63.0" \
    "google-cloud-bigquery-storage>=2.30.0,<3.0.0" \
    "langgraph>=0.2.0" \
    "langchain-core>=0.3.0" \
    "langchain-openai>=0.2.0" \
    "vllm>=0.6.0" \
    "pyyaml>=6.0" \
    "json-repair>=0.30.0" \
    "openpyxl>=3.1.0" \
    "tabulate>=0.9.0" \
    "thefuzz>=0.22.0" \

print("✅ Cài đặt dependencies thành công không có lỗi dependency conflicts!")

INFO: pip is looking at multiple versions of google-cloud-bigquery-storage to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 1.3 MB/s eta 0:00:00a 0:00:01
INFO: pip is looking at multiple versions of nvidia-cutlass-dsl-libs-base to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multipl

In [ ]:
import subprocess
import time
import requests
import os
from kaggle_secrets import UserSecretsClient

# ==========================================
# CẤU HÌNH GIAO TIẾP ĐA GPU & BỘ NHỚ CHO KAGGLE
# ==========================================
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn" 

# 🟢 Tắt lõi V1 của vLLM để tránh lỗi treo Shared Memory (/dev/shm) trên Kaggle Docker
os.environ["VLLM_USE_V1"] = "0"

# ==========================================
# 1. XÁC THỰC HUGGING FACE
# ==========================================
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("✅ Đã xác thực thành công Hugging Face Token từ Kaggle Secrets!")
except Exception as e:
    print("⚠️ Cảnh báo: Không thể lấy 'HF_TOKEN' từ Kaggle Secrets.")

# ==========================================
# 2. CẤU HÌNH VÀ KHỞI ĐỘNG vLLM
# ==========================================
MODEL_ID = "Qwen/Qwen3.5-4B"
VLLM_PORT = 8000

vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "2",      # Chia đều model cho 2 GPU
    "--dtype", "float16",               # T4 tối ưu với float16
    "--gpu-memory-utilization", "0.85", # Giữ lại 15% VRAM cho PyTorch runtime
    "--max-model-len", "4096",          # Giới hạn context window
    "--enforce-eager"                   # Tắt CUDA graphs
]

print("🚀 Đang khởi động vLLM Server trên đa GPU (V0 Engine)...")

out_file = open("vllm_stdout.log", "w")
err_file = open("vllm_stderr.log", "w")

vllm_process = subprocess.Popen(
    vllm_cmd, 
    stdout=out_file, 
    stderr=err_file
)

print("⏳ Bắt đầu nạp model vào 2 GPU VRAM (có thể mất 3-5 phút)...\n")
print("-" * 50)
ready = False
log_pos = 0  

# ==========================================
# 3. THEO DÕI LOG VÀ KIỂM TRA TRẠNG THÁI
# ==========================================
for i in range(300):
    
    if os.path.exists("vllm_stderr.log"):
        with open("vllm_stderr.log", "r") as f:
            f.seek(log_pos)          
            new_logs = f.read()      
            log_pos = f.tell()       
            if new_logs:
                print(new_logs, end="", flush=True)

    if vllm_process.poll() is not None:
        print("\n❌ Lỗi: Tiến trình vLLM đã bị crash đột ngột!")
        break

    try:
        r = requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=2)
        if r.status_code == 200:
            print("-" * 50)
            print(f"\n✅ vLLM Server đã sẵn sàng tại port {VLLM_PORT}!")
            ready = True
            break
    except Exception:
        pass
        
    time.sleep(2)

out_file.close()
err_file.close()

if not ready:
    print("\n❌ Lỗi: vLLM Server không thể khởi động thành công trong thời gian cho phép.")

✅ Đã xác thực thành công Hugging Face Token từ Kaggle Secrets!
🚀 Đang khởi động vLLM Server trên đa GPU (V0 Engine)...
⏳ Bắt đầu nạp model vào 2 GPU VRAM (có thể mất 3-5 phút)...

--------------------------------------------------
(APIServer pid=880) [transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
(APIServer pid=880) [transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
(Worker pid=934) <frozen importlib._bootstrap_external>:1301: FutureWarnin